In [59]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, expr, when, trim, round, lower, replace, min, max
from functools import reduce
from pyspark.sql.types  import IntegerType, StringType, FloatType
from pyspark.sql.window import Window


In [22]:
spark = SparkSession.builder.appName("student_mini_project").master("local[*]").getOrCreate()
sc = spark.sparkContext

print(sc)

<SparkContext master=local[*] appName=student_mini_project>


In [23]:
df_stud = spark.read.csv(
    "/content/Student_project.csv",
    header = True,
    inferSchema = True  )

df_stud.show(truncate=False)
df_stud.printSchema()
df_stud.count()

+----------+----------+---------+------+-------------+---------------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|email                      |phone     |city     |department_id|admission_date|
+----------+----------+---------+------+-------------+---------------------------+----------+---------+-------------+--------------+
|1         |Divya     |Kumar    |Male  |2003-01-31   |divya.kumar1@example.com   |6599310825|Bangalore|109          |2022-10-15    |
|2         |Karthik   |Gupta    |Male  |2004-09-24   |karthik.gupta2@example.com |6402418010|Mumbai   |104          |2021-08-05    |
|3         |Hari      |Gupta    |Male  |2000-04-18   |hari.gupta3@example.com    |9075280817|chennai  |107          |2024-07-24    |
|4         |Anitha    |Nair     |Male  |2006-08-10   |anitha.nair4@example.com   |9259052811|Hyderabad|107          |2022-12-22    |
|5         |NULL      |Patel    |Female|2001-09-28   |rohit.patel5@ex

123

In [24]:
df_null_values = df_stud.select(
    *[
        sum(when(col(c).isNull() , 1).otherwise(0)).alias(c)
        for c in df_stud.columns
    ]
)
df_null_values.show()
df_null_values.printSchema()
df_null_values.count()

+----------+----------+---------+------+-------------+-----+-----+----+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|email|phone|city|department_id|admission_date|
+----------+----------+---------+------+-------------+-----+-----+----+-------------+--------------+
|         0|         4|        0|     0|            0|    5|    3|   0|            0|             0|
+----------+----------+---------+------+-------------+-----+-----+----+-------------+--------------+

root
 |-- student_id: long (nullable = true)
 |-- first_name: long (nullable = true)
 |-- last_name: long (nullable = true)
 |-- gender: long (nullable = true)
 |-- date_of_birth: long (nullable = true)
 |-- email: long (nullable = true)
 |-- phone: long (nullable = true)
 |-- city: long (nullable = true)
 |-- department_id: long (nullable = true)
 |-- admission_date: long (nullable = true)



1

In [25]:
df_null_records = df_stud.filter(
    reduce(
        lambda a,b : a|b,
        [col(c).isNull() for c in df_stud.columns]
    )
    )
df_null_records.show()
df_null_records.count()

+----------+----------+---------+------+-------------+--------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|               email|     phone|     city|department_id|admission_date|
+----------+----------+---------+------+-------------+--------------------+----------+---------+-------------+--------------+
|         5|      NULL|    Patel|Female|   2001-09-28|rohit.patel5@exam...|6438989805|Bangalore|          107|    2022-08-15|
|         8|     Vimal|     Iyer|  Male|   2000-10-11|                NULL|9320303242|    Delhi|          102|    2021-09-02|
|        10|     Kavya|    Singh|Female|   2002-05-08|kavya.singh10@exa...|      NULL|Bangalore|          110|    2025-03-04|
|        13|      NULL|   Sharma|Female|   2002-05-14|suresh.sharma13@e...|6913224047|    Kochi|          107|    2024-08-05|
|        20|    Suresh|      Das|  Male|   2002-01-01|                NULL|9738848790|    Delhi|          109|    2024

12

In [26]:
df_not_null_records = df_stud.filter(
    reduce(
        lambda a,b : a&b,
        [col(c).isNotNull() for c in df_stud.columns]
    )
    )
df_not_null_records.show()
df_not_null_records.count()

+----------+----------+---------+------+-------------+--------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|               email|     phone|     city|department_id|admission_date|
+----------+----------+---------+------+-------------+--------------------+----------+---------+-------------+--------------+
|         1|     Divya|    Kumar|  Male|   2003-01-31|divya.kumar1@exam...|6599310825|Bangalore|          109|    2022-10-15|
|         2|   Karthik|    Gupta|  Male|   2004-09-24|karthik.gupta2@ex...|6402418010|   Mumbai|          104|    2021-08-05|
|         3|      Hari|    Gupta|  Male|   2000-04-18|hari.gupta3@examp...|9075280817|  chennai|          107|    2024-07-24|
|         4|    Anitha|     Nair|  Male|   2006-08-10|anitha.nair4@exam...|9259052811|Hyderabad|          107|    2022-12-22|
|         6|     Divya|    Singh|Female|   2003-11-09|divya.singh6@exam...|9466589567|BANGALORE|          108|    2024

111

In [27]:
from pyspark.sql.functions.builtin import initcap
df_cleaning_1 = df_not_null_records.withColumn(
    "first_name",
    initcap(trim(col("first_name")))
).withColumn(
    "last_name",
    initcap(trim(col("last_name")))
).withColumn(
    "email",
    lower(trim(col("email")))
).withColumn(
    "city",
    initcap(trim(col("city")))
).withColumn(
    "gender",
    initcap(trim(col("gender")))
)
df_cleaning_1.show(120,truncate=False)
df_cleaning_1.count()


+----------+----------+---------+------+-------------+----------------------------+----------+----------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|email                       |phone     |city      |department_id|admission_date|
+----------+----------+---------+------+-------------+----------------------------+----------+----------+-------------+--------------+
|1         |Divya     |Kumar    |Male  |2003-01-31   |divya.kumar1@example.com    |6599310825|Bangalore |109          |2022-10-15    |
|2         |Karthik   |Gupta    |Male  |2004-09-24   |karthik.gupta2@example.com  |6402418010|Mumbai    |104          |2021-08-05    |
|3         |Hari      |Gupta    |Male  |2000-04-18   |hari.gupta3@example.com     |9075280817|Chennai   |107          |2024-07-24    |
|4         |Anitha    |Nair     |Male  |2006-08-10   |anitha.nair4@example.com    |9259052811|Hyderabad |107          |2022-12-22    |
|6         |Divya     |Singh    |Female|2003-11-09   |d

111

In [28]:
df_cleaning_2 = df_cleaning_1.withColumn(
    "gender",
    when(col("gender") == "M", "Male")
    .when(col("gender") == "F", "Female")
    .otherwise(col("gender"))
)
df_cleaning_2.show(truncate=False)
df_cleaning_2.count()

+----------+----------+---------+------+-------------+--------------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|email                     |phone     |city     |department_id|admission_date|
+----------+----------+---------+------+-------------+--------------------------+----------+---------+-------------+--------------+
|1         |Divya     |Kumar    |Male  |2003-01-31   |divya.kumar1@example.com  |6599310825|Bangalore|109          |2022-10-15    |
|2         |Karthik   |Gupta    |Male  |2004-09-24   |karthik.gupta2@example.com|6402418010|Mumbai   |104          |2021-08-05    |
|3         |Hari      |Gupta    |Male  |2000-04-18   |hari.gupta3@example.com   |9075280817|Chennai  |107          |2024-07-24    |
|4         |Anitha    |Nair     |Male  |2006-08-10   |anitha.nair4@example.com  |9259052811|Hyderabad|107          |2022-12-22    |
|6         |Divya     |Singh    |Female|2003-11-09   |divya.singh6@example.c

111

In [29]:
df_duplicate_ids = df_cleaning_2.groupBy("student_id").agg(
    count("student_id").alias("student_count")
).filter(
    col("student_count") > 1
)

df_duplicate_ids.show()


+----------+-------------+
|student_id|student_count|
+----------+-------------+
|        31|            2|
|        21|            2|
|        11|            2|
+----------+-------------+



In [30]:
df_duplicate_records = df_cleaning_2.join(
    df_duplicate_ids,
    on = "student_id",
    how = "inner"
).orderBy(
    "student_id"
)

df_duplicate_records.show(truncate=False)


+----------+----------+---------+------+-------------+-------------------------+----------+------+-------------+--------------+-------------+
|student_id|first_name|last_name|gender|date_of_birth|email                    |phone     |city  |department_id|admission_date|student_count|
+----------+----------+---------+------+-------------+-------------------------+----------+------+-------------+--------------+-------------+
|11        |Sneha     |Das      |Female|2002-09-29   |sneha.das11@example.com  |7629748727|Delhi |109          |2022-05-01    |2            |
|11        |Sneha     |Das      |Female|2002-09-29   |sneha.das11@example.com  |7629748727|Delhi |109          |2022-05-01    |2            |
|21        |Swathi    |Iyer     |Male  |2001-09-18   |swathi.iyer21@example.com|8316615266|Jaipur|101          |2023-07-06    |2            |
|21        |Swathi    |Iyer     |Male  |2001-09-18   |swathi.iyer21@example.com|8316615266|Jaipur|101          |2023-07-06    |2            |
|31   

In [31]:
cleaning_stage_1 = df_cleaning_2.dropDuplicates(
    ["student_id"]
)

cleaning_stage_1.show(truncate=False)
cleaning_stage_1.count()

+----------+----------+---------+------+-------------+--------------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|email                     |phone     |city     |department_id|admission_date|
+----------+----------+---------+------+-------------+--------------------------+----------+---------+-------------+--------------+
|1         |Divya     |Kumar    |Male  |2003-01-31   |divya.kumar1@example.com  |6599310825|Bangalore|109          |2022-10-15    |
|2         |Karthik   |Gupta    |Male  |2004-09-24   |karthik.gupta2@example.com|6402418010|Mumbai   |104          |2021-08-05    |
|3         |Hari      |Gupta    |Male  |2000-04-18   |hari.gupta3@example.com   |9075280817|Chennai  |107          |2024-07-24    |
|4         |Anitha    |Nair     |Male  |2006-08-10   |anitha.nair4@example.com  |9259052811|Hyderabad|107          |2022-12-22    |
|6         |Divya     |Singh    |Female|2003-11-09   |divya.singh6@example.c

108

In [32]:
df_null_records.show()

+----------+----------+---------+------+-------------+--------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|               email|     phone|     city|department_id|admission_date|
+----------+----------+---------+------+-------------+--------------------+----------+---------+-------------+--------------+
|         5|      NULL|    Patel|Female|   2001-09-28|rohit.patel5@exam...|6438989805|Bangalore|          107|    2022-08-15|
|         8|     Vimal|     Iyer|  Male|   2000-10-11|                NULL|9320303242|    Delhi|          102|    2021-09-02|
|        10|     Kavya|    Singh|Female|   2002-05-08|kavya.singh10@exa...|      NULL|Bangalore|          110|    2025-03-04|
|        13|      NULL|   Sharma|Female|   2002-05-14|suresh.sharma13@e...|6913224047|    Kochi|          107|    2024-08-05|
|        20|    Suresh|      Das|  Male|   2002-01-01|                NULL|9738848790|    Delhi|          109|    2024

In [33]:
df_null_records_1 = df_null_records.fillna(
    {
        "first_name" : "_NA_",
        "email" : "Invalid_emails",
        "phone" : "0"

    })


df_null_records_1.show(truncate= False)


+----------+----------+---------+------+-------------+---------------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|email                      |phone     |city     |department_id|admission_date|
+----------+----------+---------+------+-------------+---------------------------+----------+---------+-------------+--------------+
|5         |_NA_      |Patel    |Female|2001-09-28   |rohit.patel5@example.com   |6438989805|Bangalore|107          |2022-08-15    |
|8         |Vimal     |Iyer     |Male  |2000-10-11   |Invalid_emails             |9320303242|Delhi    |102          |2021-09-02    |
|10        |Kavya     |Singh    |Female|2002-05-08   |kavya.singh10@example.com  |0         |Bangalore|110          |2025-03-04    |
|13        |_NA_      |Sharma   |Female|2002-05-14   |suresh.sharma13@example.com|6913224047|Kochi    |107          |2024-08-05    |
|20        |Suresh    |Das      |Male  |2002-01-01   |Invalid_emails 

In [34]:
final_cleaned_data = cleaning_stage_1.union(df_null_records_1)
final_cleaned_data.show(truncate=False)
final_cleaned_data.count()

+----------+----------+---------+------+-------------+--------------------------+----------+---------+-------------+--------------+
|student_id|first_name|last_name|gender|date_of_birth|email                     |phone     |city     |department_id|admission_date|
+----------+----------+---------+------+-------------+--------------------------+----------+---------+-------------+--------------+
|1         |Divya     |Kumar    |Male  |2003-01-31   |divya.kumar1@example.com  |6599310825|Bangalore|109          |2022-10-15    |
|2         |Karthik   |Gupta    |Male  |2004-09-24   |karthik.gupta2@example.com|6402418010|Mumbai   |104          |2021-08-05    |
|3         |Hari      |Gupta    |Male  |2000-04-18   |hari.gupta3@example.com   |9075280817|Chennai  |107          |2024-07-24    |
|4         |Anitha    |Nair     |Male  |2006-08-10   |anitha.nair4@example.com  |9259052811|Hyderabad|107          |2022-12-22    |
|6         |Divya     |Singh    |Female|2003-11-09   |divya.singh6@example.c

120

In [35]:
final_cleaned_data.write

**Summary for student_data**



*   In this student_data we have removed the 03 duplicate_vaues
("df_null_records") from the main data.
*   We cleaned the data of unwanted name casing and sorted in a Title and handel the Null value as the precious name of the columns ("final_cleaned_data")



In [36]:
df_dept = spark.read.csv(
    "/content/departments.csv",
    header = True,
    inferSchema=True
)

df_dept.show(truncate=False)
df_dept.printSchema()
df_dept.count()


+-------------+------------------------+------------+--------+----------+--------------+-------------+-----------------+
|department_id|department_name         |faculty_name|building|annual_fee|duration_years|accreditation|department_email |
+-------------+------------------------+------------+--------+----------+--------------+-------------+-----------------+
|101          |Computer Science        |Dr. Kumar   |Block A |85000     |4             |Accredited   |cs@example.com   |
|102          | information technology |Dr. Priya   |Block B |80000     |4             |Accredited   |it@example.com   |
|103          |Mechanical Engineering  |Dr. Raj     |Block C |75000     |4             |Accredited   |mech@example.com |
|104          |Civil Engineering       |NULL        |Block D |70000     |4             |Accredited   |civil@example.com|
|105          |Electrical Engineering  |Dr. Meena   |Block E |78000     |4             |Accredited   |eee@example.com  |
|106          |Electronics Engin

11

In [37]:
df_dept_duplicate = df_dept.groupBy("department_id").agg(
    count("department_id").alias("department_count")
).filter(
    col("department_count") > 1
)

df_dept_duplicate.show()

+-------------+----------------+
|department_id|department_count|
+-------------+----------------+
|          101|               2|
+-------------+----------------+



In [38]:
df_dept_cleaning_records = df_dept.dropDuplicates(
    ["department_id"]
)

df_dept_cleaning_records.show(truncate=False)
df_dept_cleaning_records.count()

+-------------+------------------------+------------+--------+----------+--------------+-------------+-----------------+
|department_id|department_name         |faculty_name|building|annual_fee|duration_years|accreditation|department_email |
+-------------+------------------------+------------+--------+----------+--------------+-------------+-----------------+
|101          |Computer Science        |Dr. Kumar   |Block A |85000     |4             |Accredited   |cs@example.com   |
|102          | information technology |Dr. Priya   |Block B |80000     |4             |Accredited   |it@example.com   |
|103          |Mechanical Engineering  |Dr. Raj     |Block C |75000     |4             |Accredited   |mech@example.com |
|104          |Civil Engineering       |NULL        |Block D |70000     |4             |Accredited   |civil@example.com|
|105          |Electrical Engineering  |Dr. Meena   |Block E |78000     |4             |Accredited   |eee@example.com  |
|106          |Electronics Engin

10

In [39]:
dept_cleaning = df_dept_cleaning_records.withColumn(
    "department_name",
    initcap(trim(col("department_name")))
)

dept_cleaning.show(truncate=False)
dept_cleaning.count()

+-------------+-----------------------+------------+--------+----------+--------------+-------------+-----------------+
|department_id|department_name        |faculty_name|building|annual_fee|duration_years|accreditation|department_email |
+-------------+-----------------------+------------+--------+----------+--------------+-------------+-----------------+
|101          |Computer Science       |Dr. Kumar   |Block A |85000     |4             |Accredited   |cs@example.com   |
|102          |Information Technology |Dr. Priya   |Block B |80000     |4             |Accredited   |it@example.com   |
|103          |Mechanical Engineering |Dr. Raj     |Block C |75000     |4             |Accredited   |mech@example.com |
|104          |Civil Engineering      |NULL        |Block D |70000     |4             |Accredited   |civil@example.com|
|105          |Electrical Engineering |Dr. Meena   |Block E |78000     |4             |Accredited   |eee@example.com  |
|106          |Electronics Engineering|D

10

In [40]:
cleaning_null = dept_cleaning.fillna("_Not Applicable_", subset=["faculty_name"])


cleaning_null.show(truncate=False)
cleaning_null.count()

+-------------+-----------------------+----------------+--------+----------+--------------+-------------+-----------------+
|department_id|department_name        |faculty_name    |building|annual_fee|duration_years|accreditation|department_email |
+-------------+-----------------------+----------------+--------+----------+--------------+-------------+-----------------+
|101          |Computer Science       |Dr. Kumar       |Block A |85000     |4             |Accredited   |cs@example.com   |
|102          |Information Technology |Dr. Priya       |Block B |80000     |4             |Accredited   |it@example.com   |
|103          |Mechanical Engineering |Dr. Raj         |Block C |75000     |4             |Accredited   |mech@example.com |
|104          |Civil Engineering      |_Not Applicable_|Block D |70000     |4             |Accredited   |civil@example.com|
|105          |Electrical Engineering |Dr. Meena       |Block E |78000     |4             |Accredited   |eee@example.com  |
|106    

10

In [41]:
cleaning_stage_2 = cleaning_null.replace(
    -1000, 1000, subset = ["annual_fee"]
)

cleaning_stage_2.show(truncate=False)
cleaning_stage_2.count()
cleaning_stage_2.printSchema()


+-------------+-----------------------+----------------+--------+----------+--------------+-------------+-----------------+
|department_id|department_name        |faculty_name    |building|annual_fee|duration_years|accreditation|department_email |
+-------------+-----------------------+----------------+--------+----------+--------------+-------------+-----------------+
|101          |Computer Science       |Dr. Kumar       |Block A |85000     |4             |Accredited   |cs@example.com   |
|102          |Information Technology |Dr. Priya       |Block B |80000     |4             |Accredited   |it@example.com   |
|103          |Mechanical Engineering |Dr. Raj         |Block C |75000     |4             |Accredited   |mech@example.com |
|104          |Civil Engineering      |_Not Applicable_|Block D |70000     |4             |Accredited   |civil@example.com|
|105          |Electrical Engineering |Dr. Meena       |Block E |78000     |4             |Accredited   |eee@example.com  |
|106    

**Play with the cleaned student_data and department_data with 20 real time scenario**

**Level 1 — Filtering and Aggregation**

Q1. Filter departments by fee — Display department ID, department name, and annual fee for departments with
annual fees greater than 70,000. Sort by annual fee in descending order.

Q2. Summarize department fees — Calculate the average, minimum, and maximum annual fee, along with the
total number of departments.



In [42]:
df_dept_filter = cleaning_stage_2.select(
    "department_id",
    "department_name",
    "annual_fee"
).filter(
    col("annual_fee") > 70000
).orderBy(
    col("annual_fee").desc()
)

df_dept_filter.show(truncate=False)


+-------------+----------------------+----------+
|department_id|department_name       |annual_fee|
+-------------+----------------------+----------+
|101          |Computer Science      |85000     |
|102          |Information Technology|80000     |
|105          |Electrical Engineering|78000     |
|103          |Mechanical Engineering|75000     |
+-------------+----------------------+----------+



In [60]:
df_dept_filter_1 = cleaning_stage_2.groupBy("department_name").agg(
    round(avg("annual_fee"),2).alias("average_fee"),
    round(min("annual_fee"),2).alias("minimum_fee"),
    round(max("annual_fee"),2).alias("maximum_fee")
)

df_dept_filter_1.show(truncate=False)


+-----------------------+-----------+-----------+-----------+
|department_name        |average_fee|minimum_fee|maximum_fee|
+-----------------------+-----------+-----------+-----------+
|Information Technology |80000.0    |80000      |80000      |
|Chemistry              |63000.0    |63000      |63000      |
|Electrical Engineering |78000.0    |78000      |78000      |
|Business Administration|65000.0    |65000      |65000      |
|Mathematics            |60000.0    |60000      |60000      |
|Physics                |62000.0    |62000      |62000      |
|Mechanical Engineering |75000.0    |75000      |75000      |
|Computer Science       |85000.0    |85000      |85000      |
|Electronics Engineering|1000.0     |1000       |1000       |
|Civil Engineering      |70000.0    |70000      |70000      |
+-----------------------+-----------+-----------+-----------+



Q3. Student count by city — Count students in each city. Sort cities by student count in descending order and
show only cities with more than five students.

Q4. Gender distribution — Count students in each gender category and calculate each category's percentage of
the total student count

In [45]:
df_stud_count = final_cleaned_data.groupBy("city").agg(
    count("student_id").alias("student_count")
).orderBy(
    col("student_count").desc()
).filter(
    col("student_count") > 5
)


df_stud_count.show(truncate=False)

+----------+-------------+
|city      |student_count|
+----------+-------------+
|Delhi     |17           |
|Bangalore |16           |
|Kochi     |15           |
|Mumbai    |13           |
|Chennai   |12           |
|Hyderabad |12           |
|Kolkata   |10           |
|Jaipur    |10           |
|Coimbatore|9            |
|Pune      |6            |
+----------+-------------+



In [46]:
df_gender_count = final_cleaned_data.groupBy("gender").agg(
    count("student_id").alias("student_count")
    ).withColumn(
        "percentage",
        round(col("student_count") / final_cleaned_data.count() * 100, 2)
    )

df_gender_count.show(truncate=False)


+------+-------------+----------+
|gender|student_count|percentage|
+------+-------------+----------+
|Female|58           |48.33     |
|Male  |62           |51.67     |
+------+-------------+----------+



**Level 2 — Conditional Logic and Data Quality**

Q5. Create fee categories — Create a fee_category column: High for fees greater than 80,000; Medium for fees
from 65,000 through 80,000; Low for fees below 65,000. Use when() and otherwise().

Q6. Admissions by year — Extract the year from admission_date, count admissions by year, and identify the year
with the most admissions.

Q7. Validate department records — Identify departments with duration_years less than or equal to zero, negative
annual fees, or invalid department email addresses. Display the records that fail validation.

In [47]:
df_dept_fee_category = cleaning_stage_2.withColumn(
    "fee_category",
    when(col("annual_fee") > 80000, "High")
    .when((col("annual_fee") >= 65000) & (col("annual_fee") <= 80000), "Medium")
    .otherwise("Low")
)

df_dept_fee_category.show(truncate=False)

+-------------+-----------------------+----------------+--------+----------+--------------+-------------+-----------------+------------+
|department_id|department_name        |faculty_name    |building|annual_fee|duration_years|accreditation|department_email |fee_category|
+-------------+-----------------------+----------------+--------+----------+--------------+-------------+-----------------+------------+
|101          |Computer Science       |Dr. Kumar       |Block A |85000     |4             |Accredited   |cs@example.com   |High        |
|102          |Information Technology |Dr. Priya       |Block B |80000     |4             |Accredited   |it@example.com   |Medium      |
|103          |Mechanical Engineering |Dr. Raj         |Block C |75000     |4             |Accredited   |mech@example.com |Medium      |
|104          |Civil Engineering      |_Not Applicable_|Block D |70000     |4             |Accredited   |civil@example.com|Medium      |
|105          |Electrical Engineering |Dr

In [48]:
year_extract = final_cleaned_data.withColumn(
    "admission_year",
    expr("year(admission_date)")
).groupBy("admission_year").agg(
    count("student_id").alias("admission_count")
).orderBy(
    col("admission_count").desc()
)

year_extract.show(truncate=False)

+--------------+---------------+
|admission_year|admission_count|
+--------------+---------------+
|2025          |28             |
|2023          |27             |
|2022          |22             |
|2024          |20             |
|2021          |14             |
|2026          |9              |
+--------------+---------------+



In [49]:
dept_records = cleaning_stage_2.filter(
    (col("duration_years") <= 0) |
    (col("department_email") == "invalid_email")
)

dept_records.show(truncate=False)

+-------------+-----------------------+------------+--------+----------+--------------+-------------+----------------+
|department_id|department_name        |faculty_name|building|annual_fee|duration_years|accreditation|department_email|
+-------------+-----------------------+------------+--------+----------+--------------+-------------+----------------+
|107          |Business Administration|Dr. Sneha   |Block G |65000     |0             |Accredited   |mba@example.com |
|109          |Physics                |Dr. Nisha   |Block I |62000     |3             |Accredited   |invalid_email   |
+-------------+-----------------------+------------+--------+----------+--------------+-------------+----------------+



**Level 3 — Joins and Relational Analysis**

Q8. Join students to departments — Perform a left join from students to departments using department_id.
Display student_id, first_name, city, department_name, faculty_name, and annual_fee.

Q9. Validate the join — Compare row counts before and after the join. Count students without a matching
department and explain whether duplicate department_id values could multiply student rows.

Q10. Count students per department — Return every department with its student count, including departments
with zero students. Choose a join direction that retains all departments.

In [50]:
Joins_tables = final_cleaned_data.join(
    cleaning_stage_2,
    on = "department_id",
    how = "left"
).select(
    "student_id",
    "first_name",
    "city",
    "department_name",
    "faculty_name",
    "annual_fee"
)

Joins_tables.show(truncate=False)
J = Joins_tables.count()
student = final_cleaned_data.count()
department = cleaning_stage_2.count()
print(f"This is the student count {student} and this would be the staff department count {department}. after joining we got {J}")



+----------+----------+---------+-----------------------+----------------+----------+
|student_id|first_name|city     |department_name        |faculty_name    |annual_fee|
+----------+----------+---------+-----------------------+----------------+----------+
|1         |Divya     |Bangalore|Physics                |Dr. Nisha       |62000     |
|2         |Karthik   |Mumbai   |Civil Engineering      |_Not Applicable_|70000     |
|3         |Hari      |Chennai  |Business Administration|Dr. Sneha       |65000     |
|4         |Anitha    |Hyderabad|Business Administration|Dr. Sneha       |65000     |
|6         |Divya     |Bangalore|Mathematics            |Dr. Hari        |60000     |
|7         |Pooja     |Kolkata  |Electronics Engineering|Dr. Arun        |1000      |
|9         |Arun      |Pune     |Mechanical Engineering |Dr. Raj         |75000     |
|11        |Sneha     |Delhi    |Physics                |Dr. Nisha       |62000     |
|12        |Anitha    |Pune     |Business Administrati

In [51]:
Joins_tables_1 = Joins_tables.groupBy(
    "department_name"
    ).agg(
    count("student_id").alias("student_count")
)
Joins_tables_1.show(truncate= False)

+-----------------------+-------------+
|department_name        |student_count|
+-----------------------+-------------+
|NULL                   |1            |
|Information Technology |12           |
|Chemistry              |13           |
|Electrical Engineering |14           |
|Business Administration|16           |
|Mathematics            |10           |
|Physics                |11           |
|Mechanical Engineering |10           |
|Computer Science       |9            |
|Electronics Engineering|9            |
|Civil Engineering      |15           |
+-----------------------+-------------+



In [52]:
Joins_tables.filter(
    col("department_name").isNull()
).show()

+----------+----------+-------+---------------+------------+----------+
|student_id|first_name|   city|department_name|faculty_name|annual_fee|
+----------+----------+-------+---------------+------------+----------+
|        41|    Anitha|Kolkata|           NULL|        NULL|      NULL|
+----------+----------+-------+---------------+------------+----------+



In [53]:
Joins_tables_cleaning = Joins_tables.fillna(
    {
        "department_name" : "_Not_listed_",
        "faculty_name" : "_Not Applicable_",
        "annual_fee" : 0
    }
)
Joins_tables_cleaning.show(120,
truncate=False)
Joins_tables_cleaning.count()

+----------+----------+----------+-----------------------+----------------+----------+
|student_id|first_name|city      |department_name        |faculty_name    |annual_fee|
+----------+----------+----------+-----------------------+----------------+----------+
|1         |Divya     |Bangalore |Physics                |Dr. Nisha       |62000     |
|2         |Karthik   |Mumbai    |Civil Engineering      |_Not Applicable_|70000     |
|3         |Hari      |Chennai   |Business Administration|Dr. Sneha       |65000     |
|4         |Anitha    |Hyderabad |Business Administration|Dr. Sneha       |65000     |
|6         |Divya     |Bangalore |Mathematics            |Dr. Hari        |60000     |
|7         |Pooja     |Kolkata   |Electronics Engineering|Dr. Arun        |1000      |
|9         |Arun      |Pune      |Mechanical Engineering |Dr. Raj         |75000     |
|11        |Sneha     |Delhi     |Physics                |Dr. Nisha       |62000     |
|12        |Anitha    |Pune      |Business 

120